# Day 3: Tree Ensembles (Random Forest & XGBoost), Benchmarking & Model Selection

**Project**: Explainable Credit Card Fraud Detection & Risk Scoring  
**Goal**: Train Random Forest and XGBoost classifiers, perform chronological hyperparameter tuning on training folds, benchmark against the Logistic Regression baseline on the validation partition, and select the optimal model candidate.

## 1. Setup & Environment Initialization
Load libraries and modular components from `src/`.

In [ ]:
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px

from src.ingestion.data_loader import load_raw_data
from src.features.data_split import create_chronological_splits
from src.models.baseline import load_model as load_baseline_model
from src.models.random_forest import train_random_forest_model, load_rf_model, get_rf_feature_importances
from src.models.xgboost_model import tune_xgboost_chronological, train_xgboost_model, load_xgb_model, get_xgb_feature_importances, compute_scale_pos_weight
from src.models.model_comparison import benchmark_models, plot_multi_model_pr_curves, plot_feature_importances_comparison

pd.set_option('display.float_format', lambda x: '%.4f' % x)
print('Environment initialized successfully.')

## 2. Load Consistent Chronological Partitions
Reusing the identical 60% Train, 20% Validation, and 20% Test chronological splits created on Day 2.

In [ ]:
df = load_raw_data(ROOT_DIR / 'data' / 'raw' / 'creditcard.csv')
train_df, val_df, test_df = create_chronological_splits(df)

X_train, y_train = train_df.drop(columns=['Class']), train_df['Class']
X_val, y_val = val_df.drop(columns=['Class']), val_df['Class']

print(f"Train : {len(train_df):,} samples | Fraud: {y_train.sum():,} ({y_train.mean()*100:.4f}%)")
print(f"Val   : {len(val_df):,} samples | Fraud: {y_val.sum():,} ({y_val.mean()*100:.4f}%)")
print(f"Test  : {len(test_df):,} samples | (Held out untouched)")

## 3. Load Baseline Logistic Regression & Train Tree Ensembles
- **Random Forest**: Trained with `class_weight='balanced_subsample'` and tree constraints (`max_depth=10`, `min_samples_leaf=2`).
- **XGBoost**: Calibrated with `scale_pos_weight` ($473.68$) and tuned via chronological TimeSeriesSplit folds.

In [ ]:
# Load baseline and trained pipelines
baseline_pipe = load_baseline_model(ROOT_DIR / 'artifacts' / 'baseline_logistic_regression.joblib')
rf_pipe = load_rf_model(ROOT_DIR / 'artifacts' / 'random_forest_pipeline.joblib')
xgb_pipe = load_xgb_model(ROOT_DIR / 'artifacts' / 'xgboost_pipeline.joblib')

models = {
    'Logistic Regression (Baseline)': baseline_pipe,
    'Random Forest': rf_pipe,
    'XGBoost': xgb_pipe,
}
print('All 3 models loaded successfully.')

## 4. Multi-Model Validation Benchmarking (Threshold = 0.5)
We benchmark all 3 models on the identical validation split.

In [ ]:
comparison_df = benchmark_models(models, X_val, y_val, threshold=0.5)
display(comparison_df[['model', 'pr_auc', 'roc_auc', 'precision', 'recall', 'f1_score', 'true_positives', 'false_positives', 'false_negatives']])

## 5. Visualizing Precision-Recall Curves
Comparing PR Curves across all three models on the same validation data.

In [ ]:
fig_dir = ROOT_DIR / 'reports' / 'figures'
plot_multi_model_pr_curves(models, X_val, y_val, output_dir=fig_dir)

from IPython.display import Image, display
display(Image(filename=str(fig_dir / '11_precision_recall_comparison.png')))

## 6. Feature Importance Analysis
Visualizing the top features driving predictions in Random Forest (Gini Importance) and XGBoost (Gain).

In [ ]:
plot_feature_importances_comparison(rf_pipe, xgb_pipe, output_dir=fig_dir, top_n=12)
display(Image(filename=str(fig_dir / '12_feature_importance_rf_xgb.png')))

## 7. Model Selection Rationale
- **XGBoost** is selected as the champion model:
  - Highest PR-AUC (**0.7613**) and ROC-AUC (**0.9827**).
  - Reduced False Positives from 1,916 to **169** (>91% reduction) while catching **82.46%** of fraud ($47/57$).
  - Provides the ideal foundation for Day 4 threshold tuning and Day 5 TreeSHAP explainability.